In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/CUAD_v1.json
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/master_clauses.csv
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/CUAD_v1_README.txt
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/full_contract_txt/WEBHELPCOMINC_03_22_2000-EX-10.8-HOSTING AGREEMENT.txt
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/full_contract_txt/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.34_11788308_EX-10.34_Sponsorship Agreement.txt
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/full_contract_txt/LOOKSMARTLTD_07_20_2012-EX-99.(D)(I)-SPONSORSHIP AGREEMENT.txt
/kaggle/input/datasets/konradb/atticus-open-contract-dataset-aok-beta/CUAD_v1/full_contract_txt/BORROWMONEYCOM,INC_06_11_2020-EX-10.1-JOINT VENTURE AGREEMENT.txt
/kaggle/input/datasets/konradb/atticus-open-contrac

In [2]:
!pip install -q "datasets==2.18.0"

In [3]:
"""
Month 2 - Step 2 (smoke test): Fine-tune LEGAL-BERT on a CUAD subset.

Purpose: validate the full training pipeline - doc-stride windowing,
null-answer (CLS-indexed) labeling, and the Trainer loop - on a small
subset before committing GPU hours to the full CUAD training set.

Run this in a Kaggle notebook with GPU accelerator enabled
(Settings > Accelerator > GPU T4 x2, and Settings > Internet > On).
"""

import numpy as np
import torch

# ---------------------------------------------------------------------------
# Step 0: GPU sanity check - fail fast before loading anything heavy
# ---------------------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. In Kaggle: Settings (right panel) > Accelerator > "
        "GPU T4 x2. Then Session > Restart & Run All."
    )

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384          # standard QA sequence length
DOC_STRIDE = 128          # overlap between windows of a long contract
SMOKE_TEST_N = 800        # small subset - just enough to validate the loop
OUTPUT_DIR = "/kaggle/working/legal-bert-cuad-smoke"

# ---------------------------------------------------------------------------
# Step 1: Load data and take a small, reproducible subset
# ---------------------------------------------------------------------------
print("Loading CUAD dataset...")
dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code=True)

train_subset = dataset["train"].shuffle(seed=42).select(range(SMOKE_TEST_N))
print(f"Smoke-test training examples: {len(train_subset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# ---------------------------------------------------------------------------
# Step 2: Feature preparation with doc-stride + null-answer (CLS) labeling
# Standard approach (mirrors HF's official run_qa.py prepare_train_features)
# ---------------------------------------------------------------------------
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        # No answer in this contract -> label points at CLS (null answer)
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = tokenized.sequence_ids(i)
        # Find the token span of the context (sequence_ids == 1)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        # If the answer isn't fully inside THIS window, label as null (CLS).
        # The answer may still be fully inside a different overlapping
        # window for the same example - that window gets the real label.
        if not (offsets[context_start][0] <= start_char and
                offsets[context_end][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= start_char:
                token_start += 1
            token_start -= 1

            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= end_char:
                token_end -= 1
            token_end += 1

            start_positions.append(token_start)
            end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


print("Tokenizing + windowing subset (this expands examples via doc-stride)...")
tokenized_train = train_subset.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_subset.column_names,
)
print(f"Expanded to {len(tokenized_train)} training features "
      f"(from {SMOKE_TEST_N} examples) after doc-stride windowing.")

# ---------------------------------------------------------------------------
# Step 3: Model + Trainer
# ---------------------------------------------------------------------------
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator,
)

print("Starting smoke-test training run...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSmoke test complete. Model saved to: {OUTPUT_DIR}")
print("If this ran without errors, the pipeline is validated.")
print("Next step: scale SMOKE_TEST_N up to the full training set "
      "and increase epochs for the real fine-tuning run.")

CUDA available: True
Device: Tesla T4
Loading CUAD dataset...


Generating train split:   0%|          | 0/22450 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4182 [00:00<?, ? examples/s]

Smoke-test training examples: 800


config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
20,2.654060
40,0.203199
60,0.074635
80,0.076384
100,0.003393
120,0.002183
140,0.116162
160,0.113108
180,0.133400
200,0.100377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Smoke test complete. Model saved to: /kaggle/working/legal-bert-cuad-smoke
If this ran without errors, the pipeline is validated.
Next step: scale SMOKE_TEST_N up to the full training set and increase epochs for the real fine-tuning run.


Pipeline validated no errors, checkpoint saved. But the numbers here reveal something important before we scale up: 800 examples expanded to 25,200 training features (3,149 steps × batch size 8) and took 38 minutes for 1 epoch. That's roughly 31 windows per contract on average — consistent with what we saw in the chunking experiment.
Extrapolated to the full CUAD training set (22,450 examples): 1 epoch would take 18 hours. That blows past Kaggle's single-session limit (~9–12h) and would eat more than half your weekly 30h GPU quota on one epoch. Scaling SMOKE_TEST_N straight up, as the script's own closing message suggested, would strand you mid-run.
The fix isn't "use less data" broadly it's targeted: most of those 31 windows per contract are null (the answer isn't in that window), since a clause typically appears in only one or two windows out of ~31. Training on every null window is mostly redundant signal. Standard practice for SQuAD2.0-style fine-tuning is to downsample null-labeled features while keeping all positive (answer-containing) ones — this cuts dataset size substantially without meaningfully hurting what the model learns.

In [ ]:

"""
Month 2 - Step 2b: Add null-answer downsampling, re-validate throughput.

The smoke test (800 examples -> ~25,200 features, 38 min/epoch) extrapolates
to ~18 hours for one epoch on the full 22,450-example CUAD train set - over
Kaggle's per-session limit. Most of those features are NULL windows (the
answer isn't in that particular window of a long contract). This script
downsamples null features while keeping all answerable ones, then reruns
on a bigger sample (3,000 examples) to get a realistic throughput estimate
before committing to the full-scale run.
"""

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected - enable Accelerator > GPU T4 x2 in Kaggle settings.")

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128
VALIDATION_N = 3000        # bigger than the smoke test, still well under full set
NEGATIVE_KEEP_RATIO = 0.25 # keep 25% of null-labeled features, 100% of answerable ones
OUTPUT_DIR = "/kaggle/working/legal-bert-cuad-validation"

print("Loading CUAD dataset...")
dataset = load_dataset("theatticusproject/cuad-qa")  # datasets==2.18.0, no trust_remote_code needed

train_subset = dataset["train"].shuffle(seed=42).select(range(VALIDATION_N))
print(f"Training examples (pre-windowing): {len(train_subset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)  # index 0 for BERT
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        sequence_ids = tokenized.sequence_ids(i)
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= start_char:
                token_start += 1
            token_start -= 1
            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= end_char:
                token_end -= 1
            token_end += 1
            start_positions.append(token_start)
            end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


print("Tokenizing + windowing...")
tokenized_train = train_subset.map(
    prepare_train_features, batched=True, remove_columns=train_subset.column_names,
)
n_before = len(tokenized_train)
print(f"Features before downsampling: {n_before}")


def downsample_nulls(tokenized_dataset, keep_ratio, seed=42):
    starts = np.array(tokenized_dataset["start_positions"])
    ends = np.array(tokenized_dataset["end_positions"])
    is_null = (starts == 0) & (ends == 0)  # CLS is always token 0 for BERT
    print(f"  Null features: {is_null.sum()} ({is_null.mean():.1%})")
    print(f"  Answerable features: {(~is_null).sum()} ({(~is_null).mean():.1%})")

    rng = np.random.default_rng(seed)
    null_indices = np.where(is_null)[0]
    n_drop = int(len(null_indices) * (1 - keep_ratio))
    drop_indices = rng.choice(null_indices, size=n_drop, replace=False)
    keep_mask = np.ones(len(tokenized_dataset), dtype=bool)
    keep_mask[drop_indices] = False
    return tokenized_dataset.select(np.where(keep_mask)[0])


tokenized_train = downsample_nulls(tokenized_train, NEGATIVE_KEEP_RATIO)
n_after = len(tokenized_train)
print(f"Features after downsampling ({NEGATIVE_KEEP_RATIO:.0%} of nulls kept): {n_after} "
      f"(reduced {1 - n_after / n_before:.1%})")

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,   # up from 8 in the smoke test - T4 should handle this at seq_len 384
    num_train_epochs=1,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=default_data_collator,
)

print("Starting validation training run (timing this for full-scale estimate)...")
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

full_train_size = 22450
projected_hours = (elapsed / VALIDATION_N) * full_train_size / 3600

print(f"\n--- Validation run complete ---")
print(f"Examples: {VALIDATION_N} -> {n_after} features after downsampling")
print(f"Elapsed: {elapsed / 60:.1f} minutes")
print(f"Projected time for full {full_train_size}-example training set, "
      f"1 epoch, same downsampling: {projected_hours:.1f} hours")
print(f"Model saved to: {OUTPUT_DIR}")